# Bonus: Week 1 pop-out experiment rebuilt in code

The grid is fixed across panels: 12 rows × 18 columns, nine target 3s, seed 1947. Panel 1 has no pop-out, Panel 2 uses colour, Panel 3 uses size only, and Panel 4 uses redundant colour plus shape. The timing chart is generated only from real timings entered for five classmates.

In [ ]:
from pathlib import Path
import csv
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

DATA = Path.cwd()
if not (DATA / 'vizlib.py').exists():
    DATA = DATA / 'Week 03'
import sys
sys.path.insert(0, str(DATA))
import vizlib

OUTPUT = DATA / 'figures'
OUTPUT.mkdir(exist_ok=True)
ROWS, COLS = 12, 18
TARGET = '3'
N_TARGETS = 9
GRID_SEED = 1947
ORDER_SEED = 2026
PANEL_NAMES = ['No channel', 'Colour', 'Size only', 'Colour + shape']
grid = None

def make_grid(seed=GRID_SEED):
    rng = np.random.default_rng(seed)
    others = [digit for digit in '0123456789' if digit != TARGET]
    result = np.array([[rng.choice(others) for _ in range(COLS)] for _ in range(ROWS)], dtype=object)
    target_positions = rng.choice(ROWS * COLS, size=N_TARGETS, replace=False)
    for position in target_positions:
        result[position // COLS, position % COLS] = TARGET
    return result

grid = make_grid()
print('Grid shape:', grid.shape, '| target count:', int((grid == TARGET).sum()))
print('Rows removed: 0; this is a controlled visual-search stimulus.')

In [ ]:
def draw_panel(axis, panel_name):
    axis.set_xlim(-0.5, COLS - 0.5)
    axis.set_ylim(-0.5, ROWS - 0.5)
    axis.invert_yaxis()
    axis.axis('off')
    axis.set_title(panel_name, fontsize=11, loc='left')
    for row in range(ROWS):
        for column in range(COLS):
            digit = grid[row, column]
            hit = digit == TARGET
            use_colour = panel_name in ['Colour', 'Colour + shape']
            use_size = panel_name == 'Size only'
            use_shape = panel_name == 'Colour + shape'
            if use_shape and hit:
                axis.add_patch(Circle((column, row), 0.40, fill=False, edgecolor='#16202E', linewidth=1.6))
            axis.text(column, row, digit, ha='center', va='center',
                      fontsize=15 if (use_size and hit) else 11,
                      fontweight='bold' if (use_size and hit) else 'normal',
                      color='#D55E00' if (use_colour and hit) else '#16202E')

fig, ax = plt.subplots(2, 2, figsize=(13, 8))
for axis, panel_name in zip(ax.flat, PANEL_NAMES):
    draw_panel(axis, panel_name)
fig.suptitle(f'Count the {TARGET}s: {N_TARGETS} targets in every panel; fixed positions and seed {GRID_SEED}',
             x=0.006, ha='left', fontsize=13.5, fontweight='bold')
fig.tight_layout(rect=[0, 0, 1, 0.94])
panel_path = OUTPUT / 'bonus_popout_panels.png'
fig.savefig(panel_path, dpi=300, bbox_inches='tight')
plt.show()
print('Saved:', panel_path)
print('Panel 3 channel: size only. Panel 4 channels: colour and shape together.')

## Timing protocol and five-person template

Show one panel at a time, use a different random order for each participant, and record seconds plus whether the participant correctly counted nine targets. Do not reuse a participant's order. The code below creates a blank CSV with 20 rows (five participants × four panels) if one does not exist. Replace the blank `seconds` and `correct` values with the actual observations; do not type invented values.

In [ ]:
timing_path = DATA / 'bonus_popout_timings.csv'
if not timing_path.exists():
    order_rng = random.Random(ORDER_SEED)
    template_rows = []
    used_orders = set()
    for participant in range(1, 6):
        order = PANEL_NAMES.copy()
        order_rng.shuffle(order)
        while tuple(order) in used_orders:
            order_rng.shuffle(order)
        used_orders.add(tuple(order))
        for position, panel_name in enumerate(order, start=1):
            template_rows.append({'participant': participant, 'presentation_order': position, 'panel': panel_name, 'seconds': '', 'correct': ''})
    pd.DataFrame(template_rows).to_csv(timing_path, index=False)
    print('Created blank timing template:', timing_path)
else:
    print('Using existing timing file:', timing_path)
timings = pd.read_csv(timing_path)
print(timings.to_string(index=False))
print('Randomization: seeded shuffle with ORDER_SEED =', ORDER_SEED, 'and a different order per participant.')

In [ ]:
required_panels = set(PANEL_NAMES)
complete = timings['seconds'].notna() & timings['correct'].notna() & (timings['seconds'].astype(str).str.strip() != '') & (timings['correct'].astype(str).str.strip() != '')
if len(timings) != 20 or set(timings['participant']) != {1, 2, 3, 4, 5} or set(timings['panel']) != required_panels:
    raise ValueError('Timing file must contain five participants and all four panels.')
if not complete.all():
    print('Timing chart is waiting for real observations: fill seconds and correct in', timing_path)
else:
    timings['seconds'] = pd.to_numeric(timings['seconds'])
    timings['correct'] = timings['correct'].astype(str).str.lower().map({'true': True, '1': True, 'yes': True, 'false': False, '0': False, 'no': False})
    if timings['seconds'].le(0).any() or timings['correct'].isna().any():
        raise ValueError('Seconds must be positive and correct must be true/false, yes/no, or 1/0.')
    summary = timings.groupby('panel', sort=False)['seconds'].agg(['mean', 'count']).reindex(PANEL_NAMES)
    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(PANEL_NAMES))
    ax.bar(x, summary['mean'], color='#0072B2', alpha=0.35, width=0.7)
    rng = np.random.default_rng(ORDER_SEED)
    for index, panel_name in enumerate(PANEL_NAMES):
        values = timings.loc[timings['panel'] == panel_name, 'seconds']
        jitter = rng.uniform(-0.16, 0.16, size=len(values))
        ax.scatter(np.full(len(values), index) + jitter, values, color='#D55E00', edgecolor='white', linewidth=0.7, zorder=3)
    ax.set_xticks(x, PANEL_NAMES)
    ax.set_ylim(0, None)
    ax.set_xlabel('Panel and pop-out channel')
    ax.set_ylabel('Time to count nine targets (seconds)')
    ax.set_title('Real timing data show which visual channel speeds target search')
    fig.text(0.01, -0.01, f'Source: five classmates (n = 5); each participant saw a seeded random panel order ({ORDER_SEED}). Bars are means; points are individual trials.', fontsize=8)
    result_path = OUTPUT / 'bonus_popout_timing_results.png'
    fig.tight_layout()
    fig.savefig(result_path, dpi=300, bbox_inches='tight')
    plt.show()
    fastest = summary['mean'].idxmin()
    slowest = summary['mean'].idxmax()
    redundant_change = summary.loc['Colour + shape', 'mean'] - summary.loc['Colour', 'mean']
    print('Fastest:', fastest)
    print('Slowest:', slowest)
    print(f'Redundant-coding change (Colour + shape minus Colour): {redundant_change:.3f} seconds')
    print('Saved:', result_path)